# Trabajo Práctico: Aprendizaje Supervisado

## Presentación del dataset

Se realizó un estudio extenso acerca de cómo los hábitos de los estudiantes, su estado psicológico y su entrega de trabajos se relacionan con la nota que obtuvieron
en el examen final. El dataset incluye:
- **Características académicas**: Como las horas de estudio, el porcentaje de entrega de trabajos, de asistencia, etc.
- **Características psicológicas**: Captura factores como el nivel de estrés, la motivación, la concentración, etc.
- **Caraceterísticas de estilo de vida digital**: Representa hábitos digitales como las horas que pasan en redes sociales.
- **Características de salud**: Duración de sueño, actividad física, etc.

Las variables objetivo son dos:
- `final_exam_score`: El resultado que obtuvieron en su último examen.
- `performance_category`: La categoría de rendimiento a la que pertenecen.

## Actividades

1. Suponga que tratamos de predecir la nota que los estudiantes obtuvieron en su examen final.
    - a) ¿De qué tipo tarea se trata? 
    - b) ¿Qué algoritmo de aprendizaje automático podríamos usar para resolverla?

a). Como la una nota es un valor continuo la tarea sería de Regresión ya que busca predecir 
b). El algoritmo más directo y comun para resolver esto es la regresion lineal,
que intenta trazar una linea matematica que pase lo mas cerca de los datos

2. Elige entre **5 y 10 columnas** que te parezcan más relevantes para predecir la nota que los estudiantes obtuvieron en su examen final. 
    **Requerimiento:** Escoge al menos una columna categórica. ¿Podemos aplicar regresión directamente con esa variable categórica?
    - a) Aplica las técnicas de procesado que creas necesarias para estas columnas categóricas. 
    - b) ¿Podemos aplicar regresión cuando hay datos nulos? Si no, aplica técnicas para manejar datos nulos y justifícalas.

In [51]:
import pandas as pd
df = pd.read_csv('student_performance_dataset.csv')
df.head()
df.isnull().sum()

student_id                      0
age                             0
gender                          0
city_type                       0
study_hours_per_day             0
deep_work_sessions              0
assignment_completion_rate      0
attendance_percentage           0
social_media_hours             60
doomscrolling_before_sleep      0
notification_distractions       0
ai_tool_usage_hours             0
gaming_hours                    0
stress_level                    0
motivation_level                0
focus_score                    60
procrastination_index           0
mental_state                    0
sleep_hours                    60
caffeine_intake                 0
physical_activity_hours         0
internet_quality                0
family_support                  0
financial_stress                0
learning_style                  0
career_goal                     0
productivity_after_midnight     0
revision_efficiency             0
burnout_risk                    0
consistency_sc

In [52]:
columnas_elegidas = [
    'study_hours_per_day',
    'attendance_percentage',
    'stress_level',
    'sleep_hours',
    'mental_state' ##jajajaj
]

# swparamos los datos de entrada (x)y lo q queremos predecir (y)
X = df[columnas_elegidas].copy()
Y= df['final_exam_score']


In [53]:

#limpiamos los datos nulos y como las horas de sueño tienen datos nulos los rellenamos con el promedio de horas de sueño de todos los estudiantes
promedio_sueno = X['sleep_hours'].mean()
X['sleep_hours'] = X['sleep_hours'].fillna(promedio_sueno)

In [54]:
# convetimos los datos de estrés a números, asignando un valor numérico a cada nivel de estrés
X = pd.get_dummies(X, columns=['mental_state'], drop_first=True)


In [55]:
X.head()

,study_hours_per_day,attendance_percentage,stress_level,sleep_hours,mental_state_Burnout,mental_state_Distracted,mental_state_Focused
0,3.2,70,8,5.8,False,False,False
1,3.9,70,4,6.1,True,False,False
2,4.3,57,2,6.6,True,False,False
3,5.3,90,8,8.0,True,False,False
4,4.1,81,4,6.9,False,False,True


3. Entrena un modelo que se ajuste a los datos y que prediga la nota que los estudiantes obtuvieron en su examen final. Sólo considera las características que mencionaste en el punto 2).
    - a) Calcula el error cuadrático medio y la raíz del error cuadrático medio. ¿Qué interpretación tiene cada métrica?
    - b) Calcula el coeficiente de determinación ($R^2$). 
    - c) ¿Qué nos indica el coeficiente en este caso? ¿Existe una relación lineal?

In [56]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, root_mean_squared_error, r2_score

In [57]:
modelo_regresion = LinearRegression()
modelo_regresion.fit(X, Y)

predicciones = modelo_regresion.predict(X)

#comparamos las notas reales (y) con las notas que adivino el modelo (predicciones)

#error cuadratico medio 
mse = mean_squared_error(Y, predicciones)

#raiz del error cuadratico

rmse = root_mean_squared_error(Y, predicciones)

#Coeficiente de determinacion
r2 = r2_score(Y, predicciones)




In [58]:
print("Resultados del modelo")
print(f"MSE(Error cuadrarico medio): {mse:.2f}")
print(f"RMSE (Raiz del error cuadratico Medio): {rmse:.2f} puntos")
print(f"R^2 (Coeficiente de determinacion): {r2:.4f}")

Resultados del modelo
MSE(Error cuadrarico medio): 255.09
RMSE (Raiz del error cuadratico Medio): 15.97 puntos
R^2 (Coeficiente de determinacion): 0.2696


a) 
MSE: representa el promedio de los errores del modelo elevados al cuadrado. Su interpretación directa es difícil porque está en una unidad irreal ("puntos al cuadrado"), pero sirve matemáticamente para penalizar o "castigar" con más fuerza a los errores muy grandes.

RMSE: Nos indica que el modelo se equivoca, en promedio, por aproximadamente 16 puntos cada vez que intenta predecir la nota del examen final de un estudiante.

b)
 El cálculo nos dio un R^2 = 0.2696 (lo que equivale a un 26.96%).

c)
-Nos indica que las características que seleccionamos (horas de estudio, asistencia, estrés, etc.) solo logran explicar casi el 27% de la nota final. El 73% restante de la nota depende de otros factores o hábitos que no incluimos en este primer intento.
-Sí, pero es una relación lineal débil. Como el valor está mucho más cerca del 0 que del 1, el modelo actual no es un buen predictor.

4. La propiedad `_coef` del modelo entrenado nos indica los **coeficientes** que el modelo ajustado le da a cada característica. 
    - a) ¿Cuál es la característica más importante según el coeficiente que se le asigna? 
    - b) ¿Cómo interpretas que uno de esos coeficientes sea negativo?

In [59]:

tabla_coeficientes = pd.DataFrame({
    'Característica': X.columns,
    'Coeficiente': modelo_regresion.coef_
})

In [60]:

tabla_coeficientes = tabla_coeficientes.sort_values(by='Coeficiente', key=abs, ascending=False)

print(" PESO DE CADA CARACTERÍSTICA ")
print(tabla_coeficientes)

 PESO DE CADA CARACTERÍSTICA 
            Característica  Coeficiente
4     mental_state_Burnout    -8.674234
6     mental_state_Focused     7.974912
0      study_hours_per_day     3.989129
3              sleep_hours     1.376295
2             stress_level    -1.245213
5  mental_state_Distracted    -0.588308
1    attendance_percentage     0.209712


b) 
- Un coeficiente negativo indica una relación inversa entre esa característica y la nota del examen. Significa que si esa variable aumenta (o está presente), la nota del alumno baja.
    - El nivel de estrés tiene un coeficiente de -1.24. Esto se interpreta como: "Por cada punto extra de estrés que tiene el alumno, su nota final baja aproximadamente 1.24 puntos".
    - En el caso del Burnout (-8.67), significa que si un alumno padece Burnout, el modelo le resta automáticamente casi 8.67 puntos a su nota final en comparación con un alumno que tiene un estado mental "Equilibrado" (Balanced).
    Por el contrario, los coeficientes positivos suman: por cada hora extra de estudio (study_hours_per_day), la nota sube 3.98 puntos.

5. Prueba con otras selecciones de características. 
    - a) Puedes agregar, quitar, o reemplazar las que hayas estudiado, o puedes seleccionar un grupo totalmente diferente (siempre un máximo de 10). 
    - b) Ajusta otro modelo de regresión a los datos seleccionados. 
    - c) Calcula las métricas (MSE, RMSE y R^2) para este nuevo modelo y compáralas con las que obtuviste antes. ¿Qué conjunto parece explicar mejor la nota 
    de los estudiantes? 

**Pista**: Ten en cuenta los coeficientes del punto 4). Además, recuerda que dos características que son **codependientes** tienden a empeorar el rendimiento de un modelo de regresión. 

In [61]:

nuevas_columnas = [
'study_hours_per_day', 
'assignment_completion_rate',
'focus_score',
'procrastination_index',
'doomscrolling_before_sleep',
'learning_style'
]

X_nuevo = df[nuevas_columnas].copy()
y = df['final_exam_score']

In [62]:

promedio_foco = X_nuevo['focus_score'].mean()
X_nuevo['focus_score'] = X_nuevo['focus_score'].fillna(promedio_foco)

X_nuevo = pd.get_dummies(X_nuevo, columns=['learning_style'], drop_first=True)

In [63]:
modelo_nuevo = LinearRegression()
modelo_nuevo.fit(X_nuevo, y)
predicciones_nuevas = modelo_nuevo.predict(X_nuevo)

In [64]:

mse_nuevo = mean_squared_error(y, predicciones_nuevas)
rmse_nuevo = root_mean_squared_error(y, predicciones_nuevas)
r2_nuevo = r2_score(y, predicciones_nuevas)


In [65]:
print("RESULTADOS DEL NUEVO MODELO (EXPERIMENTO)")
print(f"Nuevo MSE: {mse_nuevo:.2f}")
print(f"Nuevo RMSE: {rmse_nuevo:.2f} puntos")
print(f"Nuevo R²: {r2_nuevo:.4f}")

RESULTADOS DEL NUEVO MODELO (EXPERIMENTO)
Nuevo MSE: 234.47
Nuevo RMSE: 15.31 puntos
Nuevo R²: 0.3286


Al comparar los dos modelos, el nuevo conjunto de características explica mejor la nota de los estudiantes. El coeficiente de determinación (R²) subió de 0.2696 a 0.3286, lo que significa que estas nuevas variables capturan mejor el rendimiento. Además, el margen de error promedio (RMSE) bajó de 15.97 a 15.31 puntos. Esto nos demuestra que incluir variables como la entrega de trabajos y la concentración, evitando las codependientes, mejora la capacidad de predicción del modelo.

**Ejercicio extra:** Para los dos modelos que creaste, divide el dataset en un conjunto de test y otro de entrenamiento. 
    - a) Entrena nuevamente los dos modelos, solamente con el conjunto de entrenamiento.
    - b) Repite el cálculo de las métricas, pero esta vez considerando sólo el conjunto de test.
    - c) Compara los resultados utilizando la partición test/entrenamiento y utilizando el conjunto total como entrenamiento.
    - d) ¿Por qué la división entre test y entrenamiento es mejor para medir el rendimiento de un modelo? **Pista**: Si el parcial de una materia fuera exactamente igual al modelo de examen, ¿podríamos decir que el parcial sirve para medir cuánto sabe un alumno?

In [66]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_nuevo, y, test_size=0.2, random_state=42)

modelo_extra = LinearRegression()
modelo_extra.fit(X_train, y_train)

predicciones_test = modelo_extra.predict(X_test)
rmse_test = root_mean_squared_error(y_test, predicciones_test)
r2_test = r2_score(y_test, predicciones_test)


print(f"RMSE en Test: {rmse_test:.2f} puntos")
print(f"R² en Test: {r2_test:.4f}")

RMSE en Test: 14.93 puntos
R² en Test: 0.3342
